# Select H3 sequences from Nextstrain 2y metadata
Using a recent H3 HA tree, I'll pick five representative strains for the following subclades/haplotypes:
* J.2:S145N --> We probably will just use the 2024 Southern Hemisphere cell-based vaccine strain, `A/DistrictOfColumbia/27/2023`
* J.2.1 (minimally J.2:P239S, F29L)
* J.2.2 (minimally J.2:S145N, S124N)
* J.2.2 (minimally J.2.2:T65K)
* J.1.1 (minimally J.1:S145N)

In [1]:
import os
import pandas as pd

# ID input and output
datadir = '../data'
resultsdir = '../results'

os.makedirs(datadir, exist_ok=True)
os.makedirs(resultsdir, exist_ok=True)

In [2]:
metadata = pd.read_csv(os.path.join(datadir, 'flu_seasonal_h3n2_ha_2y.tsv'), sep='\t')
metadata = metadata.query('num_date>=2023') # Only keep strains appearing in 2023 or more recent

In [3]:
haplotypes = [
    # 'J.2:S145N', # We will use the 2024 Southern Hemisphere cell-based vaccine strain A/DistrictOfColumbia/27/2023
    'J.2.2:T65K'
]
subclades = [
    'J.2.1', 
    'J.2.2',
    'J.1.1'
]

def find_representative_strains(selection, column):
   
    strain_list = []
    
    for item in selection:

        item_metadata = metadata.query(f'{column} == "{item}"')

        # The early J.1.1 strains are sort of weirdly annotated in Nextstrain 2y tree
        if item == 'J.1.1':
            item_metadata = item_metadata.query('haplotype == "J.1:S145N"')
        
        lowest_divergence_strains = item_metadata[item_metadata['div'] == item_metadata['div'].min()]
        most_recent_lowest_divergence_strains = lowest_divergence_strains[lowest_divergence_strains['num_date'] == lowest_divergence_strains['num_date'].max()]

        # Just get the first strain in the dataframe if there are multiple
        most_recent_lowest_divergence_strains = most_recent_lowest_divergence_strains.head(1)
    
        # strain_list.append([item, most_recent_lowest_divergence_strains.name.values[0]])
        # Get row as list
        strain_list.append(most_recent_lowest_divergence_strains.iloc[0].tolist())


    return strain_list

print('finding representative strains...')
representative_strains = pd.DataFrame(find_representative_strains(haplotypes, 'haplotype') + 
                                      find_representative_strains(subclades, 'subclade'),
                                      columns = metadata.columns)

representative_strains = pd.concat([representative_strains, metadata.query('name == "A/DistrictOfColumbia/27/2023"')], ignore_index=True)
representative_strains

finding representative strains...


,name,div,num_date,clade_membership,subclade,haplotype
0,A/Oman/CPHL_7248310/2024,0.0464,2024.892,2a.3a.1,J.2.2,J.2.2:T65K
1,A/Shaanxi-Yuyang/1204/2024,0.0448,2024.250,2a.3a.1,J.2.1,J.2.1
2,A/Lisbon/124/2024,0.0442,2024.092,2a.3a.1,J.2.2,J.2.2
3,A/BritishColumbia/RV00920/2023,0.0476,2023.675,2a.3a.1,J.1.1,J.1:S145N
4,A/DistrictOfColumbia/27/2023,0.0442,2023.938,2a.3a.1,J.2,J.2:S145N


In [8]:
for n in representative_strains.name.unique():
    print(n)

A/Oman/CPHL_7248310/2024
A/Shaanxi-Yuyang/1204/2024
A/Lisbon/124/2024
A/BritishColumbia/RV00920/2023
A/DistrictOfColumbia/27/2023


In [ ]:
A/Lisbon/124/2024 	EPI3137180
A/BritishColumbia/RV00920/2023 EPI2813057